# Progetto H.E.R.O. - Test Esaustivo dell'Architettura Grafica

Questo notebook funge da suite di test globale per tutte le funzionalità di visualizzazione implementate nella classe `WFPInteractivePlotter`.
L'obiettivo è dimostrare le capacità del modulo di routing grafico, che delega ai sotto-moduli specializzati (Geo, TimeSeries, Distribution, Correlation) il rendering dei grafici in Altair e Plotly.

In [1]:
import sys
import os
from pathlib import Path

# Forza il path per riconoscere la cartella libs/
project_root = str(Path(os.getcwd()).resolve())
if project_root not in sys.path:
    sys.path.append(project_root)

import warnings
warnings.filterwarnings('ignore')

import altair as alt
alt.data_transformers.disable_max_rows()

from libs.multi_country_manager import MultiCountryManager
from libs.wfp_plots import WFPInteractivePlotter

print("Ambiente inizializzato.")

Ambiente inizializzato.


### Inizializzazione Architettura
Carichiamo il dataset Parquet e passiamo il manager al plotter centrale.

In [2]:
parquet_source = "data/parquet_file/wfp_consolidated.parquet"

try:
    manager = MultiCountryManager(parquet_path=parquet_source).initialize_pipeline()
    plotter = WFPInteractivePlotter(manager=manager)
    print("\n✅ Dati caricati e validati.")
except FileNotFoundError:
    print("❌ File Parquet non trovato. Assicurati di aver generato il dataset consolidato.")

2026-05-19 18:56:41,159 - libs.multi_country_manager - INFO - Caricamento rapido Parquet: wfp_consolidated.parquet
2026-05-19 18:56:44,437 - libs.multi_country_manager - WARNING - Validazione schema saltata (Pandera non attivo).
2026-05-19 18:56:44,510 - libs.multi_country_manager - INFO - Costruzione mappa dei mercati per Lazy Loading...
2026-05-19 18:56:44,673 - libs.multi_country_manager - INFO - ✅ Inizializzazione completata! 40 paesi pronti per il caricamento on-demand.
2026-05-19 18:56:44,676 - libs.wfp_plots - INFO - WFPInteractivePlotter inizializzato con 4 engine grafici.

✅ Dati caricati e validati.


In [3]:
# Estrazione automatica di tutti i codici ISO3 disponibili nel database compresso
tutti_i_paesi_disponibili = list(manager._iso3_to_name.keys())

# Generazione della mappa puntuale globale per l'anno 2026
mappa_puntuale_globale = plotter.display_geospatial_heatmap(
    iso3_list=tutti_i_paesi_disponibili,
    criterion="inflation_food_price_index",
    year=2026
)

display(mappa_puntuale_globale)

2026-05-19 18:56:44,742 - libs.wfp_plots - INFO - [GEO] Heatmap: ['AFG', 'ARM', 'BDI', 'BFA', 'BGD', 'CAF', 'CMR', 'COD', 'ETH', 'GMB', 'GNB', 'GTM', 'HTI', 'IDN', 'KEN', 'LAO', 'LBR', 'LKA', 'MDG', 'MLI', 'MMR', 'MOZ', 'MRT', 'MWI', 'NER', 'NGA', 'PHL', 'SDN', 'SEN', 'SOM', 'SSD', 'TCD', 'UGA', 'GIN', 'YEM', 'COG', 'SYR', 'IRQ', 'LBN', 'LBY'] | inflation_food_price_index | 2026/None


---
## 1. Visualizzazioni Geospaziali (Engine Plotly)
Test dei metodi mappati in `libs/plots/geo_plots.py`.

In [4]:
# 1.1 Bubble Heatmap interattiva dei mercati fisici
mappa_heatmap = plotter.display_geospatial_heatmap(
    iso3_list=["AFG", "YEM", "SOM"], 
    criterion="inflation_food_price_index", 
    year=2026
)
display(mappa_heatmap)

2026-05-19 18:56:45,618 - libs.wfp_plots - INFO - [GEO] Heatmap: ['AFG', 'YEM', 'SOM'] | inflation_food_price_index | 2026/None


In [5]:
# 1.2 Coropleta a livello nazionale (Diverging mode per inflazione)
mappa_coropleta = plotter.display_country_choropleth(
    iso3_list=[], # Lista vuota = tutti i paesi
    criterion="inflation_food_price_index", 
    year=2026, 
    diverging=True
)
display(mappa_coropleta)

2026-05-19 18:56:45,734 - libs.wfp_plots - INFO - [GEO] Choropleth: [] | inflation_food_price_index | year=2026


In [6]:
# 1.3 Strip Map Regionale (Mappa affiancata a Bar Chart del Ranking)
mappa_strip = plotter.display_regional_strip_map(
    iso3="YEM", 
    criterion="wheat", 
    year=2026, 
    max_regions=10
)
display(mappa_strip)

2026-05-19 18:56:52,798 - libs.wfp_plots - INFO - [GEO] Regional strip map: YEM | wheat
2026-05-19 18:56:52,799 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) per il paese: YEM...
2026-05-19 18:56:52,939 - libs.country_entity - INFO - Inizializzata Entità: Yemen, Rep. (YEM) -> 5174 record. Trovate 20 commodities.


In [7]:
# 1.4 Animazione temporale dell'evoluzione dei prezzi sui mercati
mappa_animata = plotter.display_market_time_animation(
    iso3="AFG", 
    criterion="wheat", 
    year=2026, 
    animate_by="month"
)
display(mappa_animata)

2026-05-19 18:56:53,048 - libs.wfp_plots - INFO - [GEO] Time animation: AFG | wheat | by=month
2026-05-19 18:56:53,049 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) per il paese: AFG...
2026-05-19 18:56:53,196 - libs.country_entity - INFO - Inizializzata Entità: Afghanistan (AFG) -> 9512 record. Trovate 4 commodities.


---
## 2. Analisi Serie Temporali (Engine Altair)
Test dei metodi mappati in `libs/plots/time_series_plots.py`.

In [8]:
# 2.1 Trend comparativo dell'inflazione con bande di confidenza
ts_inflazione = plotter.display_inflation_comparison(
    iso3_list=["AFG", "YEM", "SOM"], 
    show_confidence_band=True
)
display(ts_inflazione)

2026-05-19 18:56:53,354 - libs.wfp_plots - INFO - [TS] Inflation comparison: ['AFG', 'YEM', 'SOM']
2026-05-19 18:56:53,404 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) per il paese: SOM...
2026-05-19 18:56:53,623 - libs.country_entity - INFO - Inizializzata Entità: Somalia (SOM) -> 10672 record. Trovate 13 commodities.


alt.LayerChart(...)

In [9]:
from libs.dashboard_paese import esporta_dashboard_avanzate
esporta_dashboard_avanzate(manager=manager)

2026-05-19 18:56:53,853 - bulk_exporter - INFO - 🚀 Inizio esportazione batch per 40 paesi...
2026-05-19 18:56:53,854 - bulk_exporter - INFO - 📊 Generazione: Afghanistan (AFG)
2026-05-19 18:56:54,020 - bulk_exporter - INFO - 📊 Generazione: Armenia (ARM)
2026-05-19 18:56:54,021 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) per il paese: ARM...
2026-05-19 18:56:54,103 - libs.country_entity - INFO - Inizializzata Entità: Armenia (ARM) -> 3248 record. Trovate 24 commodities.
2026-05-19 18:56:54,204 - bulk_exporter - INFO - 📊 Generazione: Burundi (BDI)
2026-05-19 18:56:54,204 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) per il paese: BDI...
2026-05-19 18:56:54,411 - libs.country_entity - INFO - Inizializzata Entità: Burundi (BDI) -> 17400 record. Trovate 10 commodities.
2026-05-19 18:56:54,569 - bulk_exporter - INFO - 📊 Generazione: Burkina Faso (BFA)
2026-05-19 18:56:54,569 - libs.multi_country_manager - INFO - Istanziazione Lazy (on-demand) pe

In [10]:
# 2.2 Candele Giapponesi OHLC (Open-High-Low-Close) per una Commodity
ts_candele = plotter.display_commodity_candle(
    iso3="AFG", 
    commodity="wheat"
)
display(ts_candele)

2026-05-19 18:57:14,872 - libs.wfp_plots - INFO - [TS] Candlestick: AFG | wheat


alt.LayerChart(...)

In [11]:
# 2.3 Shock Heatmap (Stagionalità dell'inflazione)
ts_shock = plotter.display_shock_heatmap(iso3="YEM")
display(ts_shock)

2026-05-19 18:57:15,133 - libs.wfp_plots - INFO - [TS] Shock heatmap: YEM


alt.Chart(...)

In [12]:
# 2.4 Volatility Ribbon (Inflazione ± Deviazione Standard Mobile)
ts_volatility = plotter.display_volatility_ribbon(
    iso3="YEM", 
    window_months=3
)
display(ts_volatility)

2026-05-19 18:57:15,293 - libs.wfp_plots - INFO - [TS] Volatility ribbon: YEM | window=3
2026-05-19 18:57:15,326 - libs.plots.time_series_plots - WARNING - plot_volatility_ribbon: dati insufficienti.


SchemaValidationError: '{'data': {'name': 'empty'}}' is an invalid value.

'mark' is a required property

alt.Chart(...)

---
## 3. Analisi Distribuzionale
Test dei metodi mappati in `libs/plots/distribution_plots.py`.

In [13]:
# 3.1 Ranking Orizzontale delle Commodity più critiche
dist_ranking = plotter.display_commodity_ranking(
    iso3="AFG", 
    top_n=10, 
    reference_year=2026
)
display(dist_ranking)

2026-05-19 18:57:17,493 - libs.wfp_plots - INFO - [DIST] Commodity ranking: AFG


alt.Chart(...)

In [14]:
# 3.2 Distribuzione dei prezzi per regione (Boxplot + Strip plot combinato)
dist_prezzi = plotter.display_price_distribution(
    iso3="YEM", 
    commodity="wheat", 
    max_regions=10
)
display(dist_prezzi)

2026-05-19 18:57:17,762 - libs.wfp_plots - INFO - [DIST] Price distribution: YEM | wheat


alt.LayerChart(...)

In [15]:
# 3.3 Scatter multi-variato (Es. Inflazione vs Affidabilità dati)
dist_scatter = plotter.display_country_scatter(
    iso3_list=["AFG", "YEM", "SOM", "ETH"], 
    x_metric="inflation_food_price_index", 
    y_metric="data_coverage", 
    year=2026
)
display(dist_scatter)

2026-05-19 18:57:17,866 - libs.wfp_plots - INFO - [DIST] Country scatter: inflation_food_price_index vs data_coverage


alt.LayerChart(...)

In [16]:
# 3.4 Mosaic Chart della copertura del mercato per i top N paesi
dist_mosaic = plotter.display_market_coverage(
    iso3_list=None, 
    year=2026, 
    top_n_countries=15
)
display(dist_mosaic)

2026-05-19 18:57:19,671 - libs.wfp_plots - INFO - [DIST] Market coverage mosaic: 2026


alt.Chart(...)

---
## 4. Correlazione e Feature Selection (Machine Learning)
Test dei metodi mappati in `libs/plots/correlation_plots.py`.

In [17]:
# 4.1 Matrice di Correlazione dei Beni Primari
corr_matrix = plotter.display_correlation_matrix(
    iso3="AFG", 
    commodity_list=["wheat", "rice", "bread", "oil", "sugar"], 
    method="pearson"
)
display(corr_matrix)

2026-05-19 18:57:34,741 - libs.wfp_plots - INFO - [CORR] Correlation matrix: AFG | pearson


alt.LayerChart(...)

In [18]:
# 4.2 Analisi Lead-Lag (Anticipazione Temporale di Prezzo tra due Commodity)
corr_leadlag = plotter.display_lead_lag(
    iso3="AFG", 
    commodity_a="wheat", 
    commodity_b="bread", 
    max_lag=6
)
display(corr_leadlag)

2026-05-19 18:57:34,950 - libs.wfp_plots - INFO - [CORR] Lead-lag: AFG | wheat vs bread


alt.LayerChart(...)

In [19]:
# 4.3 Overview del Momentum (Derivata/Accelerazione del prezzo) per le Top Commodity
corr_momentum = plotter.display_momentum_overview(
    iso3="YEM", 
    top_n=6
)
display(corr_momentum)

2026-05-19 18:57:35,030 - libs.wfp_plots - INFO - [CORR] Momentum overview: YEM


alt.Chart(...)